In [1]:
# === 1. Import thư viện ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

import os
from docx import Document
import re

d:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# === 2. Đọc dữ liệu CSV đã gắn nhãn ===
df = pd.read_csv("src_data/all_labeled_questions.csv")  # 🔹 đường dẫn tới file bạn tải lên

# Kiểm tra dữ liệu
print(df.head())

# === 3. Ánh xạ nhãn thành số ===
label_map = {"question": 0, "answer": 1}
df["label_id"] = df["label"].map(label_map)

# === 4. Chia tập train/test ===
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

# === 5. Chuẩn bị tokenizer ===
model_name = "vinai/phobert-base"  # ⚠️ Nếu dữ liệu là tiếng Việt, nên đổi thành "bert-base-multilingual-cased" hoặc "PhoBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['content'], truncation=True, padding='max_length', max_length=128)

# === 6. Tạo dataset cho HuggingFace ===
train_dataset = Dataset.from_pandas(train_df[['content', 'label_id']])
test_dataset = Dataset.from_pandas(test_df[['content', 'label_id']])

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# === 7. Load mô hình gốc ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_map))

# === 8. Cấu hình huấn luyện ===
training_args = TrainingArguments(
    output_dir="./model_output",
    do_eval=True,   # để mô hình vẫn chạy đánh giá trên eval_dataset
    save_total_limit=2,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # bật FP16 nếu có GPU
    report_to="none"
)

# === 9. Huấn luyện ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# === 10. Lưu mô hình đã fine-tune ===
save_path = "/content/bert_question_answer_classifier"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Fine-tune hoàn tất. Mô hình đã lưu tại:", save_path)
print("Label map:", label_map)


In [2]:
# === 2. Hàm đọc đoạn văn từ DOCX ===
def extract_paragraphs_from_docx(docx_path):
    doc = Document(docx_path)
    return [p.text.strip() for p in doc.paragraphs if p.text.strip()]

In [6]:
text = extract_paragraphs_from_docx(r"D:\_code\DATN\data\test_data\01_TdtntTayNguyen_Ntt.docx")  # 🔹 đường dẫn tới file bạn tải lên

In [15]:
import zipfile
import os
import json
from lxml import etree
from docx import Document

# Để chuyển MathML sang LaTeX có thể dùng sympy hoặc hàm tùy chỉnh.
# from sympy.parsing.mathml import parse_mathml
# from sympy import latex

OMML_NS = {
    'm': 'http://schemas.openxmlformats.org/officeDocument/2006/math',
    'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main',
}

def extract_docx_content(docx_path, out_folder="extracted"):
    # Chuẩn bị output
    os.makedirs(out_folder, exist_ok=True)
    doc = Document(docx_path)
    results = []

    # Đọc XML gốc để lấy OMML và images
    with zipfile.ZipFile(docx_path) as zf:
        word_xml = zf.read("word/document.xml")
        doc_tree = etree.fromstring(word_xml)

        # 1. Lấy hình ảnh
        images_info = []
        media_path = os.path.join(out_folder, "media")
        os.makedirs(media_path, exist_ok=True)

        for file in zf.namelist():
            if file.startswith("word/media/"):
                img_data = zf.read(file)
                img_name = os.path.basename(file)
                img_path = os.path.join(media_path, img_name)
                with open(img_path, "wb") as f:
                    f.write(img_data)
                images_info.append({"image_name": img_name, "image_path": img_path})

        # 2. Lấy câu hỏi, đoạn văn, công thức OMML từ XML document.xml
        for p in doc_tree.findall(".//w:p", namespaces=OMML_NS):
            # Lấy text
            text_parts = [t.text for t in p.findall(".//w:t", namespaces=OMML_NS) if t.text]
            para_text = "".join(text_parts).strip()

            # Lấy công thức
            omml_nodes = p.findall(".//m:oMath", namespaces=OMML_NS) + p.findall(".//m:oMathPara", namespaces=OMML_NS)
            math_list = []

            for node in omml_nodes:
                omml_xml = etree.tostring(node, encoding="unicode")
                # OMML -> MathML có thể dùng XSLT tương tự bạn đã làm; ví dụ chuyển OMML đơn giản:
                # mathml_str = your_transform(node) # Tự định nghĩa

                math_list.append({
                    "omml": omml_xml,
                    # "mathml": mathml_str, # Nếu transform được
                    # "latex": latex_code,  # Nếu parse ra được
                })

            results.append({
                "paragraph": para_text,
                "math": math_list,
            })

        # 3. Lấy caption hoặc text liên quan đến image (nếu câu hỏi gắn với ảnh)
        # Nên duyệt từng paragraph trong docx, lấy hình ảnh kế bên hoặc kế tiếp/preceding, xử lý thủ công nếu cần

        result = {
            "path": docx_path,
            "paragraphs": results,
            "images": images_info,
        }
        out_json = os.path.join(out_folder, os.path.basename(docx_path) + ".json")
        with open(out_json, "w", encoding="utf-8") as fp:
            json.dump(result, fp, ensure_ascii=False, indent=2)
        return result

# ==== Ví dụ dùng cho nhiều file ====
if __name__ == "__main__":
    folder = "test_data"
    for f in os.listdir(folder):
        if f.endswith(".docx"):
            res = extract_docx_content(os.path.join(folder, f), out_folder="parse_output")
            print("Parsed:", f, "->", res["path"])

Parsed: 01_TdtntTayNguyen_Ntt.docx -> test_data\01_TdtntTayNguyen_Ntt.docx
Parsed: 03_Thpt-Victory_ThptNguyenVanCu-LongChau.docx -> test_data\03_Thpt-Victory_ThptNguyenVanCu-LongChau.docx
Parsed: 07_ThptCbq_ThptTdn-PhongBaoBui.docx -> test_data\07_ThptCbq_ThptTdn-PhongBaoBui.docx
Parsed: 08ThptTranHungDao-ThptChuVanAn-PhamThanh.docx -> test_data\08ThptTranHungDao-ThptChuVanAn-PhamThanh.docx
Parsed: 1.ThptHoangViet_NguyenMinhTri.docx -> test_data\1.ThptHoangViet_NguyenMinhTri.docx
Parsed: 10_TruongThptDtntN_TrangLong_ThptVietDuc-HuychungNguyen.docx -> test_data\10_TruongThptDtntN_TrangLong_ThptVietDuc-HuychungNguyen.docx
Parsed: 12_Thpt_EaSup(DeNopL2)-KhongtenKhong.docx -> test_data\12_Thpt_EaSup(DeNopL2)-KhongtenKhong.docx
Parsed: 22_Thpt_Px_Thpt_Bd-VanDao.docx -> test_data\22_Thpt_Px_Thpt_Bd-VanDao.docx
Parsed: 32_TtGdnn-GdtxM_Drak_ThptLak-ThuHa.docx -> test_data\32_TtGdnn-GdtxM_Drak_ThptLak-ThuHa.docx
Parsed: 36_Thpt-Cumgar_Thpt-PhamVanDong-LieuVoThiThuy.docx -> test_data\36_Thpt-Cum

FileNotFoundError: [Errno 2] No such file or directory: 'parse_output\\media\\'

ModuleNotFoundError: No module named 'sympy.parsing.mathml'

In [ ]:
from lxml import etree
from sympy.parsing.mathml import parse_mathml
from sympy import latex

# 1. Convert OMML XML to MathML XML
def omml_to_mathml(omml_xml, xslt_path="omml2mathml.xsl"):
    xslt_root = etree.parse(xslt_path)
    transform = etree.XSLT(xslt_root)
    omml_tree = etree.fromstring(omml_xml.encode("utf-8"))
    mathml_tree = transform(omml_tree)
    return str(mathml_tree)

# 2. Convert MathML to LaTeX
def mathml_to_latex(mathml_str):
    try:
        expr = parse_mathml(mathml_str)
        return latex(expr)
    except Exception as e:
        print("Error:", e)
        return ""

# --- Ví dụ dùng ---
omml_xml = """<m:oMath xmlns:m=\"http://schemas.openxmlformats.org/officeDocument/2006/math\" xmlns:wpc=\"http://schemas.microsoft.com/office/word/2010/wordprocessingCanvas\" xmlns:cx=\"http://schemas.microsoft.com/office/drawing/2014/chartex\" xmlns:cx1=\"http://schemas.microsoft.com/office/drawing/2015/9/8/chartex\" xmlns:mc=\"http://schemas.openxmlformats.org/markup-compatibility/2006\" xmlns:o=\"urn:schemas-microsoft-com:office:office\" xmlns:r=\"http://schemas.openxmlformats.org/officeDocument/2006/relationships\" xmlns:v=\"urn:schemas-microsoft-com:vml\" xmlns:wp14=\"http://schemas.microsoft.com/office/word/2010/wordprocessingDrawing\" xmlns:wp=\"http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing\" xmlns:w10=\"urn:schemas-microsoft-com:office:word\" xmlns:w=\"http://schemas.openxmlformats.org/wordprocessingml/2006/main\" xmlns:w14=\"http://schemas.microsoft.com/office/word/2010/wordml\" xmlns:w15=\"http://schemas.microsoft.com/office/word/2012/wordml\" xmlns:w16se=\"http://schemas.microsoft.com/office/word/2015/wordml/symex\" xmlns:wpg=\"http://schemas.microsoft.com/office/word/2010/wordprocessingGroup\" xmlns:wpi=\"http://schemas.microsoft.com/office/word/2010/wordprocessingInk\" xmlns:wne=\"http://schemas.microsoft.com/office/word/2006/wordml\" xmlns:wps=\"http://schemas.microsoft.com/office/word/2010/wordprocessingShape\"><m:r><m:rPr><m:sty m:val=\"p\"/></m:rPr><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>Δ</m:t></m:r><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>E=</m:t></m:r><m:d><m:dPr><m:ctrlPr><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:i/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr></m:ctrlPr></m:dPr><m:e><m:sSub><m:sSubPr><m:ctrlPr><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:i/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr></m:ctrlPr></m:sSubPr><m:e><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>m</m:t></m:r></m:e><m:sub><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>Po</m:t></m:r></m:sub></m:sSub><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>-</m:t></m:r><m:sSub><m:sSubPr><m:ctrlPr><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:i/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr></m:ctrlPr></m:sSubPr><m:e><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>m</m:t></m:r></m:e><m:sub><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>α</m:t></m:r></m:sub></m:sSub><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>-</m:t></m:r><m:sSub><m:sSubPr><m:ctrlPr><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:i/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr></m:ctrlPr></m:sSubPr><m:e><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>m</m:t></m:r></m:e><m:sub><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>x</m:t></m:r></m:sub></m:sSub></m:e></m:d><m:sSup><m:sSupPr><m:ctrlPr><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:i/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr></m:ctrlPr></m:sSupPr><m:e><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>c</m:t></m:r></m:e><m:sup><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>2</m:t></m:r></m:sup></m:sSup><m:r><w:rPr><w:rFonts w:ascii=\"Cambria Math\" w:eastAsia=\"Times New Roman\" w:hAnsi=\"Cambria Math\" w:cs=\"Times New Roman\"/><w:noProof/><w:kern w:val=\"0\"/><w:sz w:val=\"24\"/><w:szCs w:val=\"24\"/><w:lang w:val=\"vi-VN\"/><w14:ligatures w14:val=\"none\"/></w:rPr><m:t>=5,92(MeV)</m:t></m:r></m:oMath>"""
mathml_str = omml_to_mathml(omml_xml, xslt_path="omml2mathml.xsl")
latex_formula = mathml_to_latex(mathml_str)

print(latex_formula)  # Hiển thị công thức LaTeX đẹptext

ModuleNotFoundError: No module named 'sympy.parsing.mathml'

In [ ]:
# === 1. Load mô hình PhoBERT đã fine-tune ===
model_path = "./bert_question_answer_classifier"
clf = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path,  # ✅ dùng tokenizer của PhoBERT fine-tune
    return_all_scores=False,
    device=0,  # GPU = 0, CPU = -1
    batch_size=32   # ⭐ Tối ưu GPU
)



# === 3. Hàm xử lý từng file ===
def classify_docx(docx_path):
    paragraphs = extract_paragraphs_from_docx(docx_path)
    label_mapping = {"LABEL_0": "question", "LABEL_1": "answer"}

    classified = []
    for text in paragraphs:
        try:
            pred = clf(text)[0]
            label = label_mapping.get(pred["label"], pred["label"])
            classified.append({"text": text, "label": label})
        except Exception as e:
            classified.append({"text": text, "label": "error"})
            print(f"Lỗi khi xử lý: {text[:40]} -> {e}")

    # Gom nhóm question → answer
    grouped = []
    current_question = None
    current_answers = []

    for item in classified:
        text = item["text"]
        label = item["label"]

        if label == "question":
            if current_question and current_answers:
                grouped.append({
                    "question": current_question,
                    "answer": current_answers
                })
            current_question = text
            current_answers = []

        elif label == "answer":
            if not re.match(r"^[A-D][\.\)]", text):
                prefix = chr(65 + len(current_answers)) + ". "
                text = prefix + text
            current_answers.append(text)

    if current_question and current_answers:
        grouped.append({
            "question": current_question,
            "answer": current_answers
        })

    # Trả về DataFrame
    if grouped:
        df = pd.DataFrame(grouped)
        df["answer"] = df["answer"].apply(lambda x: str(x))
        return df
    else:
        print(f"⚠️ Không tìm thấy question/answer trong file: {os.path.basename(docx_path)}")
        return pd.DataFrame(columns=["question", "answer"])

# === 4. Xử lý nhiều file DOCX và gộp thành 1 CSV ===
input_folder = "test_data/"  # 📂 thư mục chứa các file .docx của bạn
output_csv = "src_data/all_questions_combined_2.csv"

# lấy tất cả file .docx
docx_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".docx")]
print(f"📄 Tìm thấy {len(docx_files)} file DOCX.")

# xử lý từng file
all_dataframes = []
for path in docx_files:
    print(f"🔹 Đang xử lý: {os.path.basename(path)}")
    df = classify_docx(path)
    if not df.empty:
        df["source_file"] = os.path.basename(path)
        all_dataframes.append(df)

# gộp tất cả
if all_dataframes:
    final_df = pd.concat(all_dataframes, ignore_index=True)
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"✅ Hoàn tất! Tổng {len(final_df)} câu hỏi. File lưu tại: {output_csv}")
else:
    print("⚠️ Không có dữ liệu hợp lệ nào được xuất.")


Device set to use cuda:0


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


: 